# 30 · Multimodal RAG

> 真实文档不只是字：图表、截图、扫描件、视频、语音都承载信息。多模态 RAG 让“看图/听声”也进入检索与生成。

**本文件覆盖知识点**：Text RAG / Image RAG / PDF RAG / Table RAG / Video RAG / Audio RAG / Multimodal Embedding / Vision LLM

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 多模态从哪来、到哪去

```text
PDF
 ├─ 文字  → 文本 chunk（常规向量）
 ├─ 图片  → 单独存，配图注或走多模态向量
 └─ 表格  → 结构化成 markdown/行对（Table RAG）
检索 → 按需把图片一起送给 Vision LLM 解读
```

两条技术路线：

| 路线 | 做法 | 优点 |
|------|------|------|
| **文本化** | 给图片生成 caption / 用 OCR+VL 模型把图“翻译”成文字索引 | 复用现有文本 RAG 栈 |
| **真多模态向量** | 用 CLIP/Qwen-VL 等把“图”“文”编码进同一向量空间 | 图像本身可检索 |


In [ ]:
# 知识点·真调说明：Table RAG 结构化入库 —— 把“行被拆散”的表格文本重排成可单独成 chunk 的 Markdown
_table_raw = '商品:星云基础版|类型:公有云SaaS|单价:998；商品:星云专业版|类型:私有化|单价:3999；商品:星云企业版|类型:私有化|单价:29999'
print('PDF 里抠出来的“糊成一团”的表格文本：')
print(' ', _table_raw)
print()
_llm_live(
    prompt='把下面这行“糊成一团的表格文本”整理成一个 Markdown 表格（表头为 商品/类型/单价），'
           '每行一条记录，保持数值不变：\n' + _table_raw,
    system='你是表格结构化助手：只输出一个干净的 Markdown 表格，不要输出其它解释。',
    fallback='未配置 Key 的固定样例：\n'
             '| 商品 | 类型 | 单价 |\n'
             '| --- | --- | --- |\n'
             '| 星云基础版 | 公有云SaaS | 998 |\n'
             '| 星云专业版 | 私有化 | 3999 |\n'
             '| 星云企业版 | 私有化 | 29999 |',
    temperature=0.1,
)
print('→ 表格先“结构化”成一行一条记录，切分时才不会把一行数据拦腰截断；之后每条记录能独立入库、被按行命中。')

In [ ]:
# 一条“文本化路线”的工程示意：PDF 拆页，页里带图就给图生成 caption 一并入库
import fitz  # PyMuPDF

def parse_pdf_multimodal(path):
    """把 PDF 拆成 (文本块, 图片信息) 列表；图注环节生产用 VL 模型"""
    out = []
    with fitz.open(path) as doc:
        for pno in range(len(doc)):
            page = doc[pno]
            texts = page.get_text().strip()
            imgs = page.get_images(full=True)          # 该页图片列表
            out.append({'page': pno + 1, 'text': texts,
                        'images': len(imgs),
                        'img_hint': f'第{pno+1}页含{len(imgs)}张图，可调用视觉模型生成图注后入向量库'})
    return out

print('多模态解析骨架就绪（视觉模型图注/多模态向量为进阶接入点）。')
print('将任意 PDF 放入 data/ 后调用 parse_pdf_multimodal 即可查看 页/文字/图片 结构。')

## 2. 各模态要点

| 模态 | 处理要点 |
|------|---------|
| **Image** | caption / CLIP 检索；问答时把图交给 Vision LLM |
| **Table** | 结构化为 markdown 或“表头+行”成对入 chunk，别切碎 |
| **PDF** | 页级拆解，图/表/文本分开归档（对应 05 课解析） |
| **Video** | 抽帧 + 音频转写 + 场景分段 |
| **Audio** | ASR 转写后走文本 RAG；需带时间戳便于回溯 |



In [ ]:
# 知识点·真调说明：Table RAG 读表作答 —— 检索命中结构化表格块后，文本 LLM 才能做跨行聚合
_md = ('| 商品 | 类型 | 单价 |\n'
       '| --- | --- | --- |\n'
       '| 星云基础版 | 公有云SaaS | 998 |\n'
       '| 星云专业版 | 私有化 | 3999 |\n'
       '| 星云企业版 | 私有化 | 29999 |')
print('入库的结构化表格：')
print(_md)
print()
_llm_live(
    prompt='下面是产品目录表格。\n' + _md + '\n请问：价格最高的商品是哪个？私有化产品有几个？请逐条说明依据。',
    system='你是依据表格作答的助手：只依据给定表格的数据回答，可做简单的读表/跨行聚合，不要编造。',
    fallback='未配置 Key 的固定样例：\n'
             '价格最高的商品是星云企业版（单价 29999）。私有化产品有 2 个：星云专业版、星云企业版。',
    temperature=0.1,
)
print('→ 检索命中“结构化表格块”后，普通文本 LLM 也能读表/聚合；若当初把一行切成了碎片，这类问题就答不了。'
      '图片/表格这类非纯文本模态的通用做法正是：先“文本化/结构化”，再复用文本 RAG 的检索与生成。')

## 3. 生成侧：Vision LLM

检索可能同时带回“文字片段 + 图片”。让支持视觉的大模型（如 qwen-vl）看图作答：

```text
user: [图片]这张架构图里 RAG 分几层？      ← 图直接作为消息内容
```

## 小结

- 多模态 RAG = 多模态**解析/向量** + Vision LLM **生成**；
- 起步用“文本化”（图转 caption）最省事，进阶再上真多模态向量；
- 表格/图片要单独保管，别在切分里弄丢。